# 02 — Parameter Sweeps

Sweep key parameters of the current model (fundamentalist sensitivity, trend-follower
sensitivity, the fundamentalist share) and see how they affect excess kurtosis and
return volatility. Each value is run on several seeds with `run_experiment`, which uses
all per-step returns (the same series as the README table).

In [ ]:
import sys
import os
sys.path.insert(0, '..')
os.makedirs('../figures', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from market_abm import DEFAULT_PARAMS
from market_abm.analytics import run_experiment

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

SEEDS = [1, 2, 3]


def sweep(param, values, **fixed):
    rows = []
    for v in values:
        for seed in SEEDS:
            p = {**DEFAULT_PARAMS, 'steps': 2000, 'n_agents': 100,
                 'seed': seed, param: v, **fixed}
            r = run_experiment(p)
            if r.get('valid'):
                rows.append({param: v, 'seed': seed,
                             'kurtosis': r['kurtosis'],
                             'std_return': r['volatility']})
    return pd.DataFrame(rows)


def boxes(df, param, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(data=df, x=param, y='kurtosis', ax=axes[0])
    axes[0].set_title(f'Excess Kurtosis vs {title}')
    axes[0].axhline(0, color='red', linestyle='--', alpha=0.5, label='Normal')
    axes[0].legend()
    sns.boxplot(data=df, x=param, y='std_return', ax=axes[1])
    axes[1].set_title(f'Return Volatility vs {title}')
    plt.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

## Sweep 1: Fundamentalist sensitivity

In [ ]:
df_fs = sweep('fundamental_sensitivity', [0.5, 1.0, 2.0, 4.0, 8.0])
print('Fundamental-sensitivity sweep complete.')

In [ ]:
boxes(df_fs, 'fundamental_sensitivity', 'fundamental_sensitivity', '../figures/sweep_fundamental_sensitivity.png')

## Sweep 2: Trend-follower sensitivity

In [ ]:
df_ts = sweep('trend_sensitivity', [0.0, 1.0, 5.0, 10.0, 20.0])
print('Trend-sensitivity sweep complete.')

In [ ]:
boxes(df_ts, 'trend_sensitivity', 'trend_sensitivity', '../figures/sweep_trend_sensitivity.png')

## Sweep 3: Fundamentalist fraction

In [ ]:
# frac_trend fixed at 0.05; the rest of the population is noise traders
df_frac = sweep('frac_fundamental', [0.1, 0.3, 0.5, 0.7, 0.9], frac_trend=0.05)
print('Fraction sweep complete.')

In [ ]:
boxes(df_frac, 'frac_fundamental', 'fundamentalist fraction', '../figures/sweep_fractions.png')